# Demo of the nanodent package

In [ ]:
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import numpy as np

import nanodent

In [ ]:
# Load study by passing foldet with experimental data
study = nanodent.load_folder("../tests/data/")

# Filter out experiments with low quality
study = study.classify_quality()

In [ ]:
# Manually disable experiments
study = study.disable_experiments(
  [
      "experiment_c",
  ],
  reason="manual",
)

In [ ]:
# Show timeline: grouped by maximum time gap between the experiments
timeline_fig, timeline_ax = nanodent.plot_group_timeline(study, max_gap=timedelta(minutes=200))

In [ ]:
# Or group manually by datetime ranges
datetime_ranges = [
        (
            datetime(2026, 3, 3, 0, 0, 0),
            datetime(2026, 3, 5, 0, 0, 0),
        ),
        (
            datetime(2026, 3, 5, 0, 0, 1),
            datetime(2026, 3, 12, 0, 0, 0),
        ),
]
groups = study.group_by_datetime_ranges(datetime_ranges)
timeline_fig, timeline_ax = nanodent.plot_group_timeline(groups)

In [ ]:
# Detect onset
study = study.detect_onset()

# Detect force peaks
study = study.detect_force_peaks()

# Define Savitzky-Golay filter params: window length and polynomial order
smoothing = {"window_length": 11, "polyorder": 1}

# Run Oliver-Pharr analysis on all enabled experiment
study = study.analyze_oliver_pharr(
    include_disabled=False,
    smoothing=smoothing,
    unloading_fraction=0.5,
    use_force_peak=True,
    epsilon=.75,
)

In [ ]:
# Illustrate experiments
fig, ax = plt.subplots()
nanodent.plot_experiments(
    ax,
    study.experiments,
    fit_kwargs={"color": "gray", "linestyle": "solid", "linewidth": "2"},
    smoothing=smoothing,
    zero_onset=True,
    x="disp_nm",
    y="force_uN",
    cmap="rainbow",
    marker=".",
    linestyle=None
)

ax.set_xbound(lower=0., upper=None)
ax.set_ybound(lower=0., upper=None)
ax.set_xlabel("Displacement h / nm")
ax.set_ylabel("Force P / μN")

In [ ]:
# Illustrate some experiments
fig = plt.figure(figsize=(7.22433,4))
gs = gridspec.GridSpec(nrows=1, ncols=2, hspace=.5, wspace=.5)

for g,G in enumerate(groups):
    ax = fig.add_subplot(gs[g])
    ax = nanodent.plot_experiments(
        ax,
        G,
        fit_kwargs={"color": "gold", "linestyle": "solid", "linewidth": "2"},
        zero_onset=True,
        x="disp_nm",
        y="force_uN",
        cmap="rainbow",
    )
    ax.set_xbound(lower=0., upper=None)
    ax.set_ybound(lower=0., upper=None)
    ax.set_xlabel("Displacement h / nm")
    ax.set_ylabel("Force P / μN")

In [ ]:
# Save separate plots for each experiment for inspection, optionally with Oliver-Pharr stiffness fit
nanodent.save_experiment_plots(study, output_dir = "./output", fit_kwargs={"color": "lime", "alpha": .9}, zero_onset=False)